<a href="https://colab.research.google.com/github/AmishaPuri/Bank-Loan-Risk-Dashboard/blob/main/Risk_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Set random seed for reproducibility
np.random.seed(42)
n_samples = 5000

# 2. Generate realistic feature variables
borrower_income = np.random.normal(65000, 25000, n_samples).clip(20000, 250000)
dti_ratio = np.random.beta(2, 5, n_samples) * 0.8  # Debt-to-income ratio capped at 80%
interest_rate = np.random.uniform(0.05, 0.24, n_samples)
credit_utilization = np.random.beta(3, 3, n_samples)  # Credit card utilization
delinquencies_6m = np.random.poisson(0.3, n_samples)
current_balance = np.random.uniform(5000, 45000, n_samples)

# Calculate log-odds to create a realistic default dependency structure
log_odds = (
    -2.5
    - (borrower_income / 100000) * 0.8
    + dti_ratio * 4.5
    + interest_rate * 6.0
    + credit_utilization * 3.5
    + delinquencies_6m * 1.2
)
prob = 1 / (1 + np.exp(-log_odds))
default_flag = np.random.binomial(1, prob)

# 3. Construct DataFrame
df = pd.DataFrame({
    'account_id': np.arange(1001, 1001 + n_samples),
    'borrower_income': np.round(borrower_income, 2),
    'dti_ratio': np.round(dti_ratio, 4),
    'interest_rate': np.round(interest_rate, 4),
    'credit_utilization': np.round(credit_utilization, 4),
    'delinquencies_6m': delinquencies_6m,
    'current_balance': np.round(current_balance, 2),
    'default_flag': default_flag
})

# 4. Generate local directory folder assets inside Colab
os.makedirs('data', exist_ok=True)
os.makedirs('sql', exist_ok=True)
os.makedirs('python', exist_ok=True)
os.makedirs('powerbi', exist_ok=True)

# 5. Export dataset
df.to_csv('data/raw_loans.csv', index=False)

# 6. Create SQL Schema text file directly inside Colab environment
sql_schema = """-- Schema definition for staging raw loan application profiles
CREATE TABLE raw_loans (
    account_id INT PRIMARY KEY,
    borrower_income DECIMAL(12, 2) NOT NULL,
    dti_ratio DECIMAL(5, 4) NOT NULL,
    interest_rate DECIMAL(5, 4) NOT NULL,
    credit_utilization DECIMAL(5, 4) NOT NULL,
    delinquencies_6m INT DEFAULT 0,
    current_balance DECIMAL(12, 2) NOT NULL,
    default_flag INT CHECK (default_flag IN (0, 1))
);"""

with open('sql/schema.sql', 'w') as f:
    f.write(sql_schema)

print("🎉 Step 1 Complete! Data folders, raw_loans.csv, and schema.sql are ready in Colab.")

🎉 Step 1 Complete! Data folders, raw_loans.csv, and schema.sql are ready in Colab.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

# 1. Load the raw dataset generated in Step 1
df = pd.read_csv('data/raw_loans.csv')

# 2. Separate target flag from tracking and metric features
features = ['borrower_income', 'dti_ratio', 'interest_rate', 'credit_utilization', 'delinquencies_6m']
X = df[features]
y = df['default_flag']

# 3. Train-Test Split (80% training matrix for modeling, 20% validation pool)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Feature Scaling (Essential for stable Logistic Regression coefficients)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Model Initialization with balanced weights to handle risk distributions
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

# 6. Evaluate Pipeline Performance
test_preds = model.predict(X_test_scaled)
test_probs = model.predict_proba(X_test_scaled)[:, 1]
roc_auc = roc_auc_score(y_test, test_probs)

print("--- 📊 ML MODEL EVALUATION PERFORMANCE ---")
print(f"ROC-AUC Performance Score: {roc_auc:.4f}")
print("\nClassification Matrix Report:")
print(classification_report(y_test, test_preds))

# 7. Generate Full-Portfolio Scoring (Live Credit Risk Inference)
X_all_scaled = scaler.transform(X)
df['probability_of_default'] = model.predict_proba(X_all_scaled)[:, 1]

# 8. Export scored records to the data folder for Power BI ingestion
df.to_csv('data/processed_loans.csv', index=False)
print("\n🎉 Model processing complete! Scored data saved as 'data/processed_loans.csv'.")

--- 📊 ML MODEL EVALUATION PERFORMANCE ---
ROC-AUC Performance Score: 0.7135

Classification Matrix Report:
              precision    recall  f1-score   support

           0       0.46      0.67      0.55       304
           1       0.82      0.66      0.73       696

    accuracy                           0.66      1000
   macro avg       0.64      0.67      0.64      1000
weighted avg       0.71      0.66      0.68      1000


🎉 Model processing complete! Scored data saved as 'data/processed_loans.csv'.
